# 05 - Otimização de Portfólio e Backtest
## Construção do portfólio a partir das previsões do modelo
A cada mês, selecionamos as ações do quintil superior (Q5),
otimizamos os pesos via Max Sharpe e simulamos o desempenho
considerando custos de transação.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

predictions = pd.read_parquet("../data/processed/predictions.parquet")
prices = pd.read_parquet("../data/raw/sp500_daily_prices.parquet")

print(f"Previsões: {predictions.shape}")
print(f"Preços: {prices.shape}")

### 5.1 Seleção mensal de ações
A cada final de mês, selecionamos as ações do quintil superior (top 20%)
segundo a previsão do modelo. Essas são as ações que o modelo espera
que tenham o melhor retorno nos próximos 21 dias.

In [ ]:
predictions["date"] = pd.to_datetime(predictions["date"])
predictions["year_month"] = predictions["date"].dt.to_period("M")

monthly_selections = {}

for ym, group in predictions.groupby("year_month"):
    last_day = group["date"].max()
    last_day_data = group[group["date"] == last_day]

    if len(last_day_data) < 20:
        continue

    threshold = last_day_data["predicted_return"].quantile(0.8)
    selected = last_day_data[last_day_data["predicted_return"] >= threshold]["ticker"].tolist()
    monthly_selections[ym] = selected

print(f"Meses com seleção: {len(monthly_selections)}")
example_month = list(monthly_selections.keys())[0]
print(f"\nExemplo ({example_month}): {len(monthly_selections[example_month])} ações")
print(monthly_selections[example_month][:10])

### 5.2 Cálculo dos pesos do portfólio
Para cada mês, calculamos os pesos usando otimização de Max Sharpe
com a biblioteca PyPortfolioOpt. Quando a otimização falha
(poucos dados, matriz singular), usamos pesos iguais como fallback.

Limitamos o peso máximo por ação em 10% para garantir diversificação.

In [ ]:
from pypfopt import expected_returns, risk_models, EfficientFrontier

daily_close = prices["close"].unstack("ticker")
daily_close.index = pd.to_datetime(daily_close.index)

portfolio_weights = {}
optimization_log = []

for ym, tickers in monthly_selections.items():
    end_date = daily_close.index[daily_close.index.to_period("M") == ym].max()
    start_date = end_date - pd.DateOffset(years=1)

    hist = daily_close.loc[start_date:end_date, tickers].dropna(axis=1)
    valid_tickers = hist.columns.tolist()

    if len(valid_tickers) < 5 or len(hist) < 126:
        weights = {t: 1.0 / len(valid_tickers) for t in valid_tickers}
        method = "equal"
    else:
        try:
            mu = expected_returns.mean_historical_return(hist)
            cov = risk_models.sample_cov(hist)

            ef = EfficientFrontier(mu, cov, weight_bounds=(0, 0.10))
            ef.max_sharpe(risk_free_rate=0.02)
            weights = ef.clean_weights()
            weights = {k: v for k, v in weights.items() if v > 0.001}

            if len(weights) < 3:
                weights = {t: 1.0 / len(valid_tickers) for t in valid_tickers}
                method = "equal_fallback"
            else:
                method = "max_sharpe"
        except Exception:
            weights = {t: 1.0 / len(valid_tickers) for t in valid_tickers}
            method = "equal_fallback"

    total = sum(weights.values())
    weights = {k: v / total for k, v in weights.items()}

    portfolio_weights[ym] = weights
    optimization_log.append({"month": ym, "method": method, "n_stocks": len(weights)})

log_df = pd.DataFrame(optimization_log)
print("Método de otimização utilizado:")
print(log_df["method"].value_counts())
print(f"\nMédia de ações por mês: {log_df['n_stocks'].mean():.1f}")

### 5.3 Simulação do backtest
Simulamos o retorno diário do portfólio:
- Rebalanceamento no primeiro dia útil de cada mês
- Custo de transação de 10 bps (0.10%) por operação (ida e volta)
- O portfólio é comprado (long only)

In [ ]:
daily_returns = daily_close.pct_change()

transaction_cost_bps = 10
tc_rate = transaction_cost_bps / 10000

portfolio_daily_returns = []
prev_weights = {}

sorted_months = sorted(portfolio_weights.keys())

for i, ym in enumerate(sorted_months):
    current_weights = portfolio_weights[ym]

    if i + 1 < len(sorted_months):
        next_ym = sorted_months[i + 1]
        month_dates = daily_returns.index[
            (daily_returns.index.to_period("M") == next_ym)
        ]
    else:
        month_dates = daily_returns.index[
            daily_returns.index > daily_close.index[daily_close.index.to_period("M") == ym].max()
        ]

    if len(month_dates) == 0:
        continue

    turnover = 0
    for ticker in set(list(current_weights.keys()) + list(prev_weights.keys())):
        old_w = prev_weights.get(ticker, 0)
        new_w = current_weights.get(ticker, 0)
        turnover += abs(new_w - old_w)

    tc_cost = turnover * tc_rate

    for j, date in enumerate(month_dates):
        tickers = list(current_weights.keys())
        w = np.array([current_weights.get(t, 0) for t in tickers])
        r = daily_returns.loc[date, tickers].fillna(0).values

        port_return = np.dot(w, r)

        if j == 0:
            port_return -= tc_cost

        portfolio_daily_returns.append({
            "date": date,
            "portfolio_return": port_return,
            "n_stocks": len(tickers),
            "turnover": turnover if j == 0 else 0
        })

    prev_weights = current_weights

port_df = pd.DataFrame(portfolio_daily_returns).set_index("date")
port_df["cumulative"] = (1 + port_df["portfolio_return"]).cumprod()

print(f"Período do backtest: {port_df.index.min()} até {port_df.index.max()}")
print(f"Dias simulados: {len(port_df)}")
print(f"Turnover médio mensal: {port_df['turnover'][port_df['turnover'] > 0].mean():.2%}")

### 5.4 Benchmark: SPY (Buy & Hold)
Comparamos com a estratégia mais simples: comprar e segurar o ETF SPY,
que replica o S&P 500.

In [ ]:
import yfinance as yf

spy = yf.download("SPY", start=port_df.index.min(), end=port_df.index.max(), auto_adjust=False)
spy_returns = spy["Close"].pct_change().dropna()
spy_returns.index = spy_returns.index.tz_localize(None)

common_dates = port_df.index.intersection(spy_returns.index)

port_df = port_df.loc[common_dates]
spy_aligned = spy_returns.loc[common_dates]

spy_cumulative = (1 + spy_aligned).cumprod()

print(f"Datas em comum: {len(common_dates)}")

### 5.5 Curva de performance comparativa

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

port_cum = (1 + port_df["portfolio_return"]).cumprod()
ax.plot(port_cum.index, port_cum.values, label="Estratégia ML", linewidth=1.5, color="green")
ax.plot(spy_cumulative.index, spy_cumulative.values, label="SPY (Buy & Hold)", linewidth=1.5, color="navy")

ax.set_ylabel("Retorno acumulado (base 1)")
ax.set_xlabel("Data")
ax.set_title("Estratégia ML vs SPY (Buy & Hold)")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../results/strategy_vs_spy.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.6 Métricas de performance

In [ ]:
def calc_metrics(returns, name="Strategy"):
    total_return = (1 + returns).prod() - 1
    n_years = len(returns) / 252
    annual_return = (1 + total_return) ** (1 / n_years) - 1
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = (annual_return - 0.02) / annual_vol
    sortino_downside = returns[returns < 0].std() * np.sqrt(252)
    sortino = (annual_return - 0.02) / sortino_downside
    cumulative = (1 + returns).cumprod()
    drawdowns = cumulative / cumulative.cummax() - 1
    max_dd = drawdowns.min()
    calmar = annual_return / abs(max_dd) if max_dd != 0 else np.inf

    return {
        "Estratégia": name,
        "Retorno total": f"{total_return:.2%}",
        "Retorno anual": f"{annual_return:.2%}",
        "Volatilidade anual": f"{annual_vol:.2%}",
        "Sharpe Ratio": f"{sharpe:.2f}",
        "Sortino Ratio": f"{sortino:.2f}",
        "Max Drawdown": f"{max_dd:.2%}",
        "Calmar Ratio": f"{calmar:.2f}"
    }

metrics_ml = calc_metrics(port_df["portfolio_return"], "ML Strategy")
metrics_spy = calc_metrics(spy_aligned, "SPY Buy & Hold")

metrics_table = pd.DataFrame([metrics_ml, metrics_spy]).set_index("Estratégia")
print(metrics_table.T)

### 5.7 Drawdown ao longo do tempo

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

port_cum = (1 + port_df["portfolio_return"]).cumprod()
port_dd = port_cum / port_cum.cummax() - 1

spy_dd = spy_cumulative / spy_cumulative.cummax() - 1

ax1.fill_between(port_dd.index, port_dd.values, 0, color="red", alpha=0.4, label="ML Strategy")
ax1.set_ylabel("Drawdown")
ax1.set_title("Drawdown: ML Strategy")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.fill_between(spy_dd.index, spy_dd.values, 0, color="navy", alpha=0.4, label="SPY")
ax2.set_ylabel("Drawdown")
ax2.set_title("Drawdown: SPY Buy & Hold")
ax2.set_xlabel("Data")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../results/drawdown_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.8 Retornos mensais (heatmap)

In [ ]:
monthly_ret = port_df["portfolio_return"].resample("ME").apply(
    lambda x: (1 + x).prod() - 1
)
monthly_ret.index = monthly_ret.index.to_period("M")

monthly_pivot = pd.DataFrame({
    "year": monthly_ret.index.year,
    "month": monthly_ret.index.month,
    "return": monthly_ret.values
})
monthly_pivot = monthly_pivot.pivot(index="year", columns="month", values="return")
monthly_pivot.columns = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
                          "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

fig, ax = plt.subplots(figsize=(14, 8))
import seaborn as sns
sns.heatmap(monthly_pivot, annot=True, fmt=".1%", cmap="RdYlGn", center=0,
            linewidths=0.5, ax=ax, cbar_kws={"label": "Retorno mensal"})
ax.set_title("Retornos mensais da estratégia ML")
ax.set_ylabel("Ano")
plt.tight_layout()
plt.savefig("../results/monthly_returns_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.9 Salvar resultados do backtest

In [ ]:
port_df.to_parquet("../data/processed/backtest_results.parquet")
metrics_table.to_csv("../results/performance_metrics.csv")

print("Arquivos salvos:")
print("  - data/processed/backtest_results.parquet")
print("  - results/performance_metrics.csv")
print(f"\nShape do backtest: {port_df.shape}")